<a id="1"></a>
# <div style="text-align:center; border-radius:15px 50px; padding:15px; color:white; margin:0; font-size:150%; font-family:Pacifico; background-color:#8B0000; overflow:hidden"><b> Thermophysical Property: Melting Point</b></div>

# 🧪 Melting Point Prediction Using Fused Molecular Representations

Predicting the melting point of organic compounds is a long-standing challenge in chemistry and chemical engineering. Melting point is a critical physicochemical property that influences drug formulation, material selection, and process safety. However, experimental measurements are often expensive, time-consuming, or unavailable, motivating the use of data-driven machine learning approaches.

In this notebook, we develop a regression pipeline to predict the melting point (Tm) of organic molecules using a **fusion of structural and group contribution features**. The dataset provides molecular SMILES strings alongside group contribution descriptors, enabling the model to learn both high-level functional group information and fine-grained structural patterns.

## 🔬 Methodology Overview

The proposed approach combines three complementary sources of information:

- **Group contribution features**  
  Precomputed subgroup counts representing functional groups within each molecule.

- **Character-level SMILES representations**  
  SMILES strings are vectorized using TF-IDF over character n-grams to capture local structural patterns and bonding syntax.

- **Token-level SMILES representations**  
  SMILES are tokenized into chemically meaningful units (atoms, bonds, ring indices, and symbols) and encoded using word-level TF-IDF to capture higher-level molecular motifs.

These feature sets are concatenated into a single sparse representation and used to train a **LightGBM regression model**, which effectively captures nonlinear relationships and feature interactions.

## ⚙️ Model Training Strategy

- The target variable (melting point) is log-transformed to stabilize variance and reduce the impact of extreme values.
- Model performance is evaluated using **5-fold cross-validation**, with Mean Absolute Error (MAE) as the evaluation metric.
- Early stopping is applied during training to prevent overfitting.
- Final predictions are obtained by averaging test predictions across folds and transforming them back to the original scale.

## 🏁 Results

This feature fusion strategy significantly improves predictive performance compared to using group descriptors alone, achieving a competitive MAE on the Kaggle leaderboard. The results demonstrate that incorporating both **chemical structure information from SMILES** and **domain-informed group descriptors** is essential for accurate melting point prediction.

---

**Keywords:** Melting Point Prediction, SMILES, TF-IDF, LightGBM, Molecular Descriptors, Feature Fusion


<a id="1"></a>
# <div style="text-align:center; border-radius:15px 50px; padding:7px; color:white; margin:0; font-size:110%; font-family:Pacifico; background-color:#3168a1; overflow:hidden"><b> Libraries </b></div>

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import mean_absolute_error
from scipy.sparse import hstack, csr_matrix

import lightgbm as lgb


In [ ]:
train = pd.read_csv("/kaggle/input/melting-point/train.csv")
test  = pd.read_csv("/kaggle/input/melting-point/test.csv")

print(train.shape, test.shape)



In [ ]:
train.head()
train.info()


<a id="1"></a>
# <div style="text-align:center; border-radius:15px 50px; padding:7px; color:white; margin:0; font-size:110%; font-family:Pacifico; background-color:#3168a1; overflow:hidden"><b> Data Cleaning & Preprocessing</b></div>

In [ ]:
y = train["Tm"].values

# Group contribution features
group_cols = [c for c in train.columns if c.startswith("Group")]
X_group_train = train[group_cols].values
X_group_test  = test[group_cols].values

# SMILES
smiles_train = train["SMILES"].astype(str)
smiles_test  = test["SMILES"].astype(str)


In [ ]:
import re

def tokenize_smiles(smiles):
    pattern = r"(\[[^\]]+\]|Br|Cl|Si|Na|Ca|Li|Mg|Al|Sn|Zn|[A-Z][a-z]?|\d+|=|#|\(|\)|\.|\+|\-)"
    tokens = re.findall(pattern, smiles)
    return " ".join(tokens)


In [ ]:
smiles_train_tok = train["SMILES"].astype(str).apply(tokenize_smiles)
smiles_test_tok  = test["SMILES"].astype(str).apply(tokenize_smiles)


In [ ]:
# Character-level
tfidf_char = TfidfVectorizer(
    analyzer="char",
    ngram_range=(3, 6),
    min_df=2,
    max_features=40000
)

# Token-level
tfidf_word = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 2),
    min_df=2,
    max_features=30000
)

X_char_train = tfidf_char.fit_transform(smiles_train)
X_char_test  = tfidf_char.transform(smiles_test)

X_word_train = tfidf_word.fit_transform(smiles_train_tok)
X_word_test  = tfidf_word.transform(smiles_test_tok)


In [ ]:
from scipy.sparse import hstack, csr_matrix

X_group_train = csr_matrix(train[group_cols].values)
X_group_test  = csr_matrix(test[group_cols].values)

X_train = hstack([X_group_train, X_char_train, X_word_train])
X_test  = hstack([X_group_test, X_char_test, X_word_test])


In [ ]:
y = np.log1p(train["Tm"].values)


<a id="1"></a>
# <div style="text-align:center; border-radius:15px 50px; padding:7px; color:white; margin:0; font-size:110%; font-family:Pacifico; background-color:#3168a1; overflow:hidden"><b> Lightgbm Model </b></div>

In [ ]:
lgb_params = {
    "objective": "regression",
    "metric": "mae",
    "learning_rate": 0.03,
    "num_leaves": 256,
    "feature_fraction": 0.7,
    "bagging_fraction": 0.7,
    "bagging_freq": 5,
    "lambda_l1": 2.0,
    "lambda_l2": 2.0,
    "verbosity": -1,
    "seed": 42
}


In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

oof = np.zeros(len(train))
test_preds = np.zeros(len(test))

for fold, (tr, val) in enumerate(kf.split(X_train)):
    print(f"\nFold {fold+1}")

    train_set = lgb.Dataset(X_train[tr], label=y[tr])
    val_set   = lgb.Dataset(X_train[val], label=y[val])

    model = lgb.train(
        lgb_params,
        train_set,
        num_boost_round=5000,
        valid_sets=[val_set],
        callbacks=[
            lgb.early_stopping(300),
            lgb.log_evaluation(200)
        ]
    )

    oof[val] = model.predict(X_train[val], num_iteration=model.best_iteration)
    test_preds += model.predict(X_test, num_iteration=model.best_iteration) / kf.n_splits

# inverse log
oof = np.expm1(oof)
test_preds = np.expm1(test_preds)

print("\nCV MAE:", mean_absolute_error(train["Tm"], oof))


<a id="1"></a>
# <div style="text-align:center; border-radius:15px 50px; padding:7px; color:white; margin:0; font-size:110%; font-family:Pacifico; background-color:#3168a1; overflow:hidden"><b> Predictions </b></div>

In [ ]:
submission = pd.DataFrame({
    "id": test["id"],
    "Tm": test_preds
})

submission.to_csv("submission.csv", index=False)
